<a href="https://colab.research.google.com/github/guthasushmasree/ML-LabPrograms/blob/main/ml(skill_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from google.colab import files




print("Upload your 50K placement prediction CSV file")

uploaded = files.upload()

filename = list(uploaded.keys())[0]

print("\nUploaded file:", filename)




df = pd.read_csv(filename)

print("\n============================================================")
print("DATASET LOADED")
print("============================================================")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())




before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\n============================================================")
print("STEP 1: REMOVE DUPLICATES")
print("============================================================")

print("Rows before:", before)
print("Rows after :", after)
print("Duplicates removed:", before - after)




print("\n============================================================")
print("STEP 2: MISSING VALUES")
print("============================================================")

print("\nMissing values BEFORE:")

print(df.isna().sum())

print(
    "\nTotal missing values BEFORE:",
    df.isna().sum().sum()
)




NUM = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]




CAT = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "CGPA_Tier"
]




df["HistoryOfBacklogs"] = df["HistoryOfBacklogs"].map({
    "Yes": 1,
    "No": 0
})



for col in NUM:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )




for col in NUM:

    df[col] = df[col].fillna(
        df[col].median()
    )


for col in CAT:

    df[col] = df[col].fillna(
        df[col].mode()[0]
    )


print("\nMissing values AFTER filling:")

print(df[NUM + CAT].isna().sum())

print(
    "\nTotal missing values AFTER:",
    df[NUM + CAT].isna().sum().sum()
)




print("\n============================================================")
print("STEP 3: CGPA OUTLIER DETECTION")
print("============================================================")


q1 = df["CGPA"].quantile(0.25)

q3 = df["CGPA"].quantile(0.75)

iqr = q3 - q1

lo = q1 - 1.5 * iqr

hi = q3 + 1.5 * iqr


outlier_mask = (
    (df["CGPA"] < lo) |
    (df["CGPA"] > hi)
)


outlier_count = int(
    outlier_mask.sum()
)


print("Q1:", round(q1, 2))

print("Q3:", round(q3, 2))

print("IQR:", round(iqr, 2))

print("Lower fence:", round(lo, 2))

print("Upper fence:", round(hi, 2))

print(
    "CGPA outliers:",
    outlier_count
)




df["CGPA"] = df["CGPA"].clip(
    lower=lo,
    upper=hi
)

print("\nCGPA outliers clipped successfully.")




prep = ColumnTransformer(

    transformers=[

        (
            "num",

            Pipeline([
                (
                    "imp",
                    SimpleImputer(
                        strategy="median"
                    )
                ),

                (
                    "sc",
                    StandardScaler()
                )
            ]),

            NUM
        ),


        (
            "cat",

            Pipeline([
                (
                    "imp",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),

                (
                    "oh",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),

            CAT
        )
    ]
)




X = df[NUM + CAT]

y = df["PlacementStatus"]


print("\n============================================================")
print("FEATURES AND TARGET")
print("============================================================")

print("Number of input features:", X.shape[1])

print("Target column:", "PlacementStatus")

print("Target values:")

print(y.value_counts())




X_processed = prep.fit_transform(X)




print("\n============================================================")
print("STEP 4: SCALING + ONE-HOT ENCODING")
print("============================================================")

print(
    "Original feature count:",
    len(NUM) + len(CAT)
)

print(
    "Processed shape:",
    X_processed.shape
)




feature_names = prep.get_feature_names_out()


print("\nFirst 20 processed feature names:")

print(
    list(feature_names[:20])
)




numeric_part = X_processed[:, :len(NUM)]


if hasattr(
    numeric_part,
    "toarray"
):

    numeric_part = numeric_part.toarray()


means = numeric_part.mean(
    axis=0
)


print(
    "\nNumeric means after scaling "
    "(should be approximately 0):"
)


print(
    np.round(
        means,
        3
    )
)




print("\n============================================================")
print("PLACEMENT STATUS DISTRIBUTION")
print("============================================================")

print(
    df["PlacementStatus"].value_counts()
)


print("\nPlacement percentage:")

print(
    df["PlacementStatus"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)




print("\n============================================================")
print("ANOMALY INFORMATION")
print("============================================================")

print(
    df["IsAnomaly"].value_counts()
)




df.to_csv(
    "placement_clean.csv",
    index=False
)


print("\n============================================================")
print("CLEAN DATASET SAVED")
print("============================================================")

print(
    "File created: placement_clean.csv"
)

print(
    "Final dataset shape:",
    df.shape
)




import joblib


joblib.dump(
    prep,
    "placement_preprocessor.joblib"
)


print(
    "Preprocessor saved as: "
    "placement_preprocessor.joblib"
)




files.download(
    "placement_clean.csv"
)


print("\n============================================================")
print("EXPERIMENT 2 COMPLETED SUCCESSFULLY!")
print("============================================================")

Upload your 50K placement prediction CSV file


Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset.csv

Uploaded file: placement_predict_50k Dataset.csv

DATASET LOADED
Dataset shape: (50000, 32)

Column names:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly', 'Salary Package']

First 5 rows:


,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly,Salary Package
0,1,Male,Ahmedabad,Tier2,ECE,Networking,No,No,6.02,6.54,...,0,66.7,2.2,49.4,47.8,0,Low,0,0,0.00
1,2,Female,Mumbai,Tier2,ECE,DataScience,Yes,Yes,5.84,5.12,...,0,48.2,2.4,26.7,25.8,0,Low,0,0,0.00
2,3,Male,Kolkata,Tier2,IT,DataScience,Yes,No,4.91,5.29,...,0,73.8,2.8,67.7,41.5,0,Low,1,0,3.89
3,4,Male,Jaipur,Tier1,CS,AI,No,No,7.67,8.03,...,0,69.8,2.7,66.9,48.0,0,Mid,1,0,8.37
4,5,Male,Pune,Tier2,IT,DataScience,Yes,No,8.14,8.97,...,1,73.1,2.1,71.7,61.7,1,High,1,0,18.99



STEP 1: REMOVE DUPLICATES
Rows before: 50000
Rows after : 50000
Duplicates removed: 0

STEP 2: MISSING VALUES

Missing values BEFORE:
StudentID                0
Gender                   0
City                     0
CollegeTier              0
Stream                   0
Specialisation           0
Hostel                   0
HistoryOfBacklogs        0
SGPA_Sem1                0
SGPA_Sem2                0
SGPA_Sem3                0
SGPA_Sem4                0
SGPA_Sem5                0
SGPA_Sem6                0
SGPA_Sem7                0
SGPA_Sem8                0
CGPA                     0
AttendancePercent        0
Internships              0
Projects                 0
Workshops             4488
Certifications           0
Publications             0
AptitudeTestScore     4029
SoftSkillsRating      3526
CodingTestScore       2976
MockInterviewScore    4957
ExtraCurricular          0
CGPA_Tier                0
PlacementStatus          0
IsAnomaly                0
Salary Package           0
d

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


EXPERIMENT 2 COMPLETED SUCCESSFULLY!
